# Cell-state generation on OpenProblems

MultiFlow is trained on the OpenProblems bone-marrow multiome data (69,249 cells,
22 cell types). It then generates paired RNA and ATAC profiles for every cell type,
and the generated cells are compared with held-out cells using the metrics of
Fig. 2 of the paper.

| Step | Function |
|---|---|
| Download and split the data | `mf.download_dataset`, `mf.split_h5mu` |
| Train the two encoders | `mf.train_autoencoder`, `mf.train_rna_vae` |
| Encode the training cells | `mf.encode` |
| Train MultiFlow | `mf.train` |
| Generate and decode cells | `mf.generate`, `mf.decode` |
| Score against held-out cells | `mf.metrics.evaluate_generation` |

Everything is trained from scratch, and the only requirement is
`pip install multiflow-omics`. Use a GPU: with the paper settings the encoders and
the flow train for several hours. To check the installation first, set
`QUICK = True`; the same steps then run on a small simulated dataset in about a
minute on a CPU. Every step writes a file and is skipped when that file exists, so
an interrupted run continues where it stopped.

In [ ]:
import os
import time
from pathlib import Path

import multiflow as mf

QUICK = os.environ.get("MULTIFLOW_QUICK") == "1"  # or simply: QUICK = True
DEVICE = None  # None uses the GPU when there is one

if QUICK:  # simulated data and small networks: about a minute on a CPU
    WORK = Path("runs/generation_quick")
    AUTOENCODER = dict(epochs=40, rna_dims=(128, 64, 16), atac_dims=(128, 64, 16))
    RNA_VAE = dict(max_steps=300)
    FLOW = dict(epochs=30, hidden_dim=64, batch_size=64)
else:  # the settings of the paper
    WORK = Path("runs/generation")
    AUTOENCODER = dict(epochs=300)
    RNA_VAE = dict(max_steps=200_000)
    FLOW = dict(epochs=1600)

SCIMILARITY_DIR = Path("external/annotation_model_v1")  # optional, see section 2

started = time.time()
print("multiflow-omics", mf.__version__)

## 1. Data

The file is 8.4 GB (Figshare, CC BY 4.0). RNA holds raw counts and ATAC is binary.
One fifth of the cells of every cell type is set aside; these cells are used only
for the metrics in section 5.

In [ ]:
if QUICK:
    data = WORK / "simulated.h5mu"
    if not data.exists():
        mf.simulate_h5mu(data, n_perturbations=0)
else:
    data = mf.download_dataset(
        "openproblem", "data/openproblem_filtered.h5mu", accept_license=True
    )

report = mf.validate_h5mu(data)
print(f"{report['n_obs']:,} cells, {report['n_conditions']} cell types")
print("raw RNA counts and binary ATAC:", report["paired_raw_contract_ok"])

train, test = WORK / "split" / "train.h5mu", WORK / "split" / "test.h5mu"
if not train.exists():
    mf.split_h5mu(data, WORK / "split", train_fraction=0.8)

## 2. Encoders

MultiFlow models 128-dimensional latent states, not the profiles themselves. For
generation, RNA is encoded by a VAE on log-normalised expression and ATAC by the
ATAC branch of the paired autoencoder. Both are trained on the training cells only.

In the paper the hidden layers of the RNA VAE start from the public SCimilarity
annotation model. To do the same, download and extract it once:

```bash
mkdir -p external
curl -L "https://zenodo.org/records/8286452/files/annotation_model_v1.tar.gz?download=1" \
  -o external/annotation_model_v1.tar.gz
tar -xzf external/annotation_model_v1.tar.gz -C external
```

Without that folder the VAE starts from random weights, which works but is not the
setting of the paper.

In [ ]:
scimilarity = SCIMILARITY_DIR if SCIMILARITY_DIR.is_dir() else None
print("RNA VAE starts from:", "SCimilarity" if scimilarity else "random weights")

autoencoder = WORK / "autoencoder" / "autoencoder.pt"
if not autoencoder.exists():
    mf.train_autoencoder(train, autoencoder.parent, device=DEVICE, **AUTOENCODER)

rna_vae = WORK / "rna_vae" / "rna_vae.pt"
if not rna_vae.exists():
    mf.train_rna_vae(
        train,
        rna_vae.parent,
        scimilarity_model_dir=scimilarity,
        device=DEVICE,
        **RNA_VAE,
    )

encoded = WORK / "train_encoded.h5mu"
if not encoded.exists():
    mf.encode(
        train,
        encoded,
        task="generation",
        autoencoder=autoencoder,
        rna_vae=rna_vae,
        device=DEVICE,
    )

## 3. Train MultiFlow

One flow per modality, coupled by cross-attention and conditioned on the cell type.

In [ ]:
run = WORK / "multiflow"
if not (run / "model.pt").exists():
    mf.train(encoded, run, model="cell-state", device=DEVICE, **FLOW)

## 4. Generate paired cells

For every cell type we generate as many cells as were held out (at most 200), with
100 integration steps, and decode them with the decoders that belong to the two
encoders. RNA comes out as log-normalised expression, ATAC as binary accessibility.

In [ ]:
held_out = mf.read_obs(test)["cell_type"].astype(str).value_counts()
n_cells = held_out.clip(upper=200).to_dict()

latent = WORK / "generated_latent.h5mu"
if not latent.exists():
    mf.generate(run, latent, n=n_cells, device=DEVICE)

profiles = WORK / "generated_profiles.h5mu"
if not profiles.exists():
    mf.decode(
        latent,
        profiles,
        task="generation",
        reference=encoded,
        autoencoder=autoencoder,
        rna_vae=rna_vae,
        device=DEVICE,
    )
print(f"{sum(n_cells.values()):,} generated cells of {len(n_cells)} cell types")

## 5. Metrics

Generated and held-out cells are matched by cell type and compared per modality:

- `pcc`, `scc`: Pearson and Spearman correlation of the mean profiles;
- `mmd`, `mmd_multiscale`: maximum mean discrepancy on 50 principal components, with
  one Gaussian kernel or with the five kernels of the scDiffusion-X evaluation (lower
  is better);
- `knn_mixing`: how evenly real and generated cells mix among the 90 nearest
  neighbours (0 to 1, higher is better);
- `ilisi`: the scIB iLISI, added when the optional package `scib` is installed.

In [ ]:
use_ilisi = mf.metrics.scib_available()
table = mf.metrics.evaluate_generation(test, profiles, ilisi=use_ilisi)
table.to_csv(WORK / "generation_metrics.csv")
table.round(4)

For comparison, Fig. 2 of the paper reports for MultiFlow:

| | PCC | SCC | MMD | iLISI |
|---|---|---|---|---|
| RNA | 0.998 | 0.992 | 0.015 | 0.90 |
| ATAC | 0.994 | 0.973 | 0.006 | 0.92 |

A new training run does not reproduce these digits exactly: the encoders are
retrained, the held-out cells differ, and MMD depends on the estimator and on the
cells that are sampled.

## 6. A look at the cells (optional)

Mean profiles of generated against held-out cells, and a joint UMAP of the RNA
profiles. Needs `matplotlib`; the UMAP also needs `scanpy`
(`pip install "multiflow-omics[notebooks]"`).

In [ ]:
import importlib.util

import mudata as mu
import numpy as np

if importlib.util.find_spec("matplotlib"):
    import matplotlib.pyplot as plt

    real, generated = mu.read_h5mu(test), mu.read_h5mu(profiles)
    real_rna = mf.metrics.log1p_cp10k(real["rna"].X)
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.4))
    panels = [
        ("RNA, mean log expression", real_rna, generated["rna"].X),
        ("ATAC, fraction of cells open", real["atac"].X, generated["atac"].X),
    ]
    for ax, (title, observed, sampled) in zip(axes, panels, strict=True):
        x = np.asarray(observed.mean(axis=0)).ravel()
        y = np.asarray(sampled.mean(axis=0)).ravel()
        ax.scatter(x, y, s=3, color="#2a6f97", alpha=0.4, linewidths=0, rasterized=True)
        limit = max(x.max(), y.max())
        ax.plot([0, limit], [0, limit], color="0.4", linewidth=0.8)
        ax.set(title=title, xlabel="held-out cells", ylabel="generated cells")
        ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(WORK / "mean_profiles.png", dpi=200)
    plt.show()

In [ ]:
if importlib.util.find_spec("matplotlib") and importlib.util.find_spec("scanpy"):
    import anndata as ad
    import scanpy as sc

    rna = ad.AnnData(np.vstack([real_rna, np.asarray(generated["rna"].X)]))
    rna.obs["source"] = ["held-out"] * real.n_obs + ["generated"] * generated.n_obs
    rna.obs["cell type"] = list(real["rna"].obs["cell_type"].astype(str)) + list(
        generated["rna"].obs["cell_type"].astype(str)
    )
    sc.pp.pca(rna, n_comps=min(50, rna.n_vars - 1))
    sc.pp.neighbors(rna)
    sc.tl.umap(rna)
    sc.pl.umap(rna, color=["source", "cell type"], wspace=0.35, frameon=False)

In [ ]:
print(f"finished in {(time.time() - started) / 3600:.2f} h")